In [1]:
import numpy as np
import matplotlib.pyplot as plt
#import scienceplots
import pandas as pd
import pickle
import os

dpi = 1600
 
# get the location of this file
# Imposta il percorso principale manualmente
main_fpath = r"C:\Users\Lorenzo\Desktop\Semester project"
print(f"Percorso principale impostato su: {main_fpath}")

 
def loader(fname, load_dir=['data','processed']):
    load_path = os.path.join(main_fpath, *load_dir, fname)
    with open(load_path, 'rb') as f:
        df = pd.read_pickle(f)
    return df
 
# Main Data:
load_path_3d = 'processed_slice_data_3d_2.pkl' # pruned random 150 dataset (coordinate slices)
load_path_2d = 'processed_slice_data_2d_2.pkl' # pruned random 150 dataset (coordinate slices)
# load_path_3d = 'slice_data_3d.pkl' # pruned random 150 dataset (coordinate slices)
# load_path_2d = 'slice_data_2d_new.pkl' # pruned random 150 dataset (coordinate slices)
 
# # Baseline Data:
load_path_scalars_3d = 'opt_scalars_df_3d.pkl' # scalars for optimization (input parameters, opt history, etc.)
load_path_scalars_2d = 'opt_scalars_df_2d.pkl' # scalars for optimization (input parameters, opt history, etc.)
 
df_3d = loader(load_path_3d)
df_scalars_3d = loader(load_path_scalars_3d, load_dir=['data','raw'])
 
df_2d = loader(load_path_2d)
df_scalars_2d = loader(load_path_scalars_2d, load_dir=['data','raw'])
 
print("Slice keys (3D):")
print(df_3d.keys())
 
print("Slice keys (2D):")
print(df_2d.keys())
 
print("Number of cases (3D):")
print(len(df_3d['case_num'].unique()))

Percorso principale impostato su: C:\Users\Lorenzo\Desktop\Semester project
Slice keys (3D):
Index(['case_num', 'slice_num', 'sub_slice_num', 'eta', 'CoordinateX',
       'CoordinateY', 'VelocityX', 'VelocityY', 'VelocityZ', 'CoefPressure'],
      dtype='object')
Slice keys (2D):
Index(['case_num', 'slice_num', 'CoordinateX', 'CoordinateY', 'VelocityX',
       'VelocityY', 'VelocityZ', 'CoefPressure'],
      dtype='object')
Number of cases (3D):
776


In [9]:
import importlib
np.random.seed(0)
rng = np.random.default_rng(13)
 
save_plot = True
save_path = os.path.join('slice_eta_8.png')
 
n_cases = len(df_3d['case_num'].unique())
# print(f'Total number of cases: {n_cases}')
 
rand_nums = rng.choice(n_cases, size=4, replace=False)
# [877, 1018, 345, 267]
# convert into a list of valid indices
id_list = df_3d['case_num'].unique()
rand_idx = [id_list[i] for i in rand_nums]
 
# Choose non-randomly (uncomment below to choose non-randomly)
# rand_idx = [4, 68, 110, 877]
# rand_idx = [877, 1220, 345, 269]
rand_idx = [4, 4]
 
# print(len(rand_idx))
# print(f'Randomly selected case numbers: {rand_idx}')
letters = ['(a)', '(b)', '(c)', '(d)']
# First plot optimized slice and initial slice
# 5 samples, top optimized, bottom initial
fig, axs = plt.subplots(2, 1, figsize=(20, 5), dpi=dpi)
z_axes = [8]
linestyles = ['-', '-', '-s', '-D', '-^']
 
for i, z_axis in enumerate(z_axes):
 
    df_3d_sample = df_scalars_3d.loc[rand_idx[i]]
    df_2d_sample = df_scalars_2d.loc[rand_idx[i]]
 
    # Get the case parameters
    case_params_2d = df_2d_sample['case_params']
    case_params = df_3d_sample['case_params']
    print('Case parameters:')
    print('3D: ', case_params)
    print('2D: ', case_params_2d)
    # mach = case_params['mach'].item()
    # reynolds = case_params['reynolds'].item()
    # cl = case_params['mycl'].item()
    # area_min = case_params['v'].item()
    # alpha_final = df_3d_sample['alpha'][-1].item()
    # Plot optimized slice (the last slice for each case)
 
    # Get the optimized slice
    max_slice_num_3d = df_3d.loc[(df_3d['case_num'] == rand_idx[i])]['slice_num'].max()
    slice_nums_3d = [max_slice_num_3d, max_slice_num_3d]
 
    max_slice_num_2d = df_2d.loc[(df_2d['case_num'] == rand_idx[i])]['slice_num'].max()
    slice_nums_2d = [max_slice_num_2d, max_slice_num_2d]
 
    # print("z value: ", df_3d.loc[(df_3d['case_num'] == rand_idx[i]) & (df_3d['slice_num'] == slice_nums[i]) & (df_3d['sub_slice_num'] == z_axis)]['CoordinateZ'].values)
    slice_opt_3d = df_3d.loc[(df_3d['case_num'] == rand_idx[i]) & (df_3d['slice_num'] == slice_nums_3d[i]) & (df_3d['sub_slice_num'] == z_axis)]
    slice_opt_2d = df_2d.loc[(df_2d['case_num'] == rand_idx[i]) & (df_2d['slice_num'] == slice_nums_2d[i])]
    # plot the x 'CoordinateX' and y 'CoordinateY' coordinates of the slice
    coordx_3d = slice_opt_3d['CoordinateX'].values
    coordy_3d = slice_opt_3d['CoordinateY'].values
    coordx_2d = slice_opt_2d['CoordinateX'].values
    coordy_2d = slice_opt_2d['CoordinateY'].values
    # Compute the distance from the y value of the first coordinate to 0
    y_dist = coordy_3d[0]
    # Compute the distance from the x value of the first coordinate to 1
    x_dist = 1 - coordx_3d[0]
    # Now shift all coordinates by the x and y distances
    coordx_3d = coordx_3d + x_dist
    coordy_3d = coordy_3d - y_dist
 
    # Recenter the coordinates to a start at (1,0)
    coordx_2d = coordx_2d - coordx_2d[0] + 1
    coordy_2d = coordy_2d - coordy_2d[0]
 
    coordx_3d = coordx_3d - coordx_3d[0] + 1
    coordy_3d = coordy_3d - coordy_3d[0]
 
    # coords_x_reordered, coords_y_reordered = funcs.reorder_coords(slice_opt)
    axs[0].plot(coordx_2d, coordy_2d, linestyles[i], linewidth=2.0, alpha=0.5, label=f'Optimized (2D) {letters[i]}', color='blue')
    axs[0].plot(coordx_3d, coordy_3d, linestyles[i], linewidth=2.0, alpha=0.5, label=f'Optimized (3D) {letters[i]}', color='red')
    # Plot the last point of the slice and the first point of the slice
    # axs[0].plot(coordx_2d[0], coordy_2d[0], 'o', color='black', alpha=0.5, label='Initial (2D)', markersize=10)
    # axs[0].plot(coordx_3d[0], coordy_3d[0], 'o', color='orange', alpha=0.5, label='Initial (3D)', markersize=10)
 
    # axs[0].plot(coordx_2d[-1], coordy_2d[-1], 'x', color='black', alpha=0.5, label='Final (2D)', markersize=10)
    # axs[0].plot(coordx_3d[-1], coordy_3d[-1], 'x', color='orange', alpha=0.5, label='Final (3D)', markersize=10)
 
    # # Add vectors to show the direction of the slice
    # axs[0].quiver(coordx_2d[:-1], coordy_2d[:-1], coordx_2d[1:]-coordx_2d[:-1], coordy_2d[1:]-coordy_2d[:-1], scale_units='xy', angles='xy', scale=1, color='red', alpha=0.5)
    # axs[0].quiver(coordx_3d[:-1], coordy_3d[:-1], coordx_3d[1:]-coordx_3d[:-1], coordy_3d[1:]-coordy_3d[:-1], scale_units='xy', angles='xy', scale=1, color='blue', alpha=0.5)
 
    coef_press_2d = slice_opt_2d['CoefPressure']
    coef_press_3d = slice_opt_3d['CoefPressure']
 
    # Plot the pressure coefficient
    axs[1].plot(coordx_2d, coef_press_2d, linestyles[i], linewidth=2.0, alpha=0.5, label=f'Optimized (2D) {letters[i]}', color='blue')
    axs[1].plot(coordx_3d, coef_press_3d, linestyles[i], linewidth=2.0, alpha=0.5, label=f'Optimized (3D) {letters[i]}', color='red')
 
    # # Plot the raw pressure coefficient
    # axs[1].plot(slice_opt_2d['CoordinateX'], slice_opt_2d['CoefPressure'], 'o', color='black', alpha=0.5, label='Raw (2D)', markersize=10)
    # axs[1].plot(slice_opt_3d['CoordinateX'], slice_opt_3d['CoefPressure'], 'o', color='orange', alpha=0.5, label='Raw (3D)', markersize=10)
 
# Remove x and y axes, and ticks
axs[0].set_xticks([])
axs[0].set_yticks([])
# axs[1].set_xticks([])
# axs[1].set_yticks([])
# Remove the top and right spines
axs[0].spines['top'].set_visible(False)
axs[0].spines['right'].set_visible(False)
axs[0].spines['bottom'].set_visible(False)
axs[0].spines['left'].set_visible(False)
axs[1].spines['top'].set_visible(False)
axs[1].spines['right'].set_visible(False)
# Set the font size
axs[0].tick_params(axis='both', which='major', labelsize=14)
axs[1].tick_params(axis='both', which='major', labelsize=14)
 
 
plt.tight_layout()
plt.legend(loc='lower right')
if save_plot:
    plt.savefig(save_path, dpi=dpi)

Case parameters:
3D:  {'mach': np.float64(0.528263150384875), 'reynolds': np.float64(1793662.0350126843), 'mycl': np.float64(1.0228501440547313), 'v': np.float64(0.8101030915676433)}
2D:  {'mach': np.float64(0.528263150384875), 'reynolds': np.float64(1793662.0350126843), 'mycl': np.float64(1.0228501440547313), 'v': np.float64(0.8101030915676433)}


In [5]:
import os

# Percorso dove salvare i CSV
export_dir = r"C:\Users\Lorenzo\Desktop\Semester project\exports"
os.makedirs(export_dir, exist_ok=True)

# Scegli le colonne principali da esportare (puoi modificarle)
cols_3d = ['case_num', 'slice_num', 'sub_slice_num', 'eta',
           'CoordinateX', 'CoordinateY', 'VelocityX', 'VelocityY', 'VelocityZ', 'CoefPressure']

cols_2d = ['case_num', 'slice_num',
           'CoordinateX', 'CoordinateY', 'VelocityX', 'VelocityY', 'VelocityZ', 'CoefPressure']

# Esporta i dataset completi (solo colonne selezionate)
df_3d[cols_3d].to_csv(os.path.join(export_dir, "data_3d_full.csv"), index=False)
df_2d[cols_2d].to_csv(os.path.join(export_dir, "data_2d_full.csv"), index=False)

print("✅ File CSV esportati con successo!")
print("📁 Percorso cartella:", export_dir)

# (Facoltativo) Mostra un’anteprima dei dati
print("\n📊 Prime righe del file 3D:")
print(df_3d[cols_3d].head())

print("\n📊 Prime righe del file 2D:")
print(df_2d[cols_2d].head())


✅ File CSV esportati con successo!
📁 Percorso cartella: C:\Users\Lorenzo\Desktop\Semester project\exports

📊 Prime righe del file 3D:
   case_num  slice_num  sub_slice_num  eta  CoordinateX  CoordinateY  \
0       0.0        0.0            0.0  0.1     0.990000     0.001751   
1       0.0        0.0            0.0  0.1     0.990000     0.002100   
2       0.0        0.0            0.0  0.1     0.990000     0.002100   
3       0.0        0.0            0.0  0.1     0.978647     0.004517   
4       0.0        0.0            0.0  0.1     0.967256     0.006925   

   VelocityX  VelocityY  VelocityZ  CoefPressure  
0  -0.000024   0.008598  -0.000029      0.009024  
1   0.005155   0.001369  -0.000014     -0.002818  
2   0.005155   0.001369  -0.000014     -0.002818  
3   0.003534  -0.000741  -0.000019      0.035632  
4   0.003838  -0.000802  -0.000019      0.017922  

📊 Prime righe del file 2D:
   case_num  slice_num  CoordinateX  CoordinateY  VelocityX  VelocityY  \
0       0.0        2.0   

In [6]:
import os

# Cartella dove salvare il CSV
export_dir = r"C:\Users\Lorenzo\Desktop\Semester project\exports"
os.makedirs(export_dir, exist_ok=True)

# ===========================
# PARAMETRI DA SCEGLIERE
# ===========================
case_number = 4        # il numero del caso che vuoi salvare
slice_number = 8       # numero dello slice

# ===========================
# SELEZIONE DELLO SLICE 2D
# ===========================
slice_df = df_2d.loc[
    (df_2d['case_num'] == case_number) &
    (df_2d['slice_num'] == slice_number),
    ['CoordinateX', 'CoordinateY']
]

filename = f"airfoil_coords_2d_case{case_number}_slice{slice_number}.csv"

# ===========================
# SALVATAGGIO
# ===========================
slice_df.to_csv(os.path.join(export_dir, filename), index=False)

print(f"✅ Slice 2D salvata con successo in CSV: {filename}")
print("📁 Percorso cartella:", export_dir)
print("\n📊 Prime righe del CSV:")
print(slice_df.head())


✅ Slice 2D salvata con successo in CSV: airfoil_coords_2d_case4_slice8.csv
📁 Percorso cartella: C:\Users\Lorenzo\Desktop\Semester project\exports

📊 Prime righe del CSV:
       CoordinateX  CoordinateY
34752     0.990000     0.000627
34753     0.990000     0.001046
34754     0.990000     0.001046
34755     0.978923     0.002955
34756     0.967861     0.004787


In [7]:
import os

# Cartella dove salvare il CSV
export_dir = r"C:\Users\Lorenzo\Desktop\Semester project\exports"
os.makedirs(export_dir, exist_ok=True)

# ===========================
# PARAMETRI DA SCEGLIERE
# ===========================
case_number = 4        # il numero del caso che vuoi salvare
slice_number = 8       # numero dello slice

# ===========================
# SELEZIONE DELLO SLICE 2D
# ===========================
slice_df = df_2d.loc[
    (df_2d['case_num'] == case_number) &
    (df_2d['slice_num'] == slice_number),
    ['CoordinateX', 'CoordinateY', 'CoefPressure']
]

filename = f"airfoil_coords_pressure_2d_case{case_number}_slice{slice_number}.csv"

# ===========================
# SALVATAGGIO
# ===========================
slice_df.to_csv(os.path.join(export_dir, filename), index=False)

print(f"✅ Slice 2D con pressione salvata in CSV: {filename}")
print("📁 Percorso cartella:", export_dir)
print("\n📊 Prime righe del CSV:")
print(slice_df.head())


✅ Slice 2D con pressione salvata in CSV: airfoil_coords_pressure_2d_case4_slice8.csv
📁 Percorso cartella: C:\Users\Lorenzo\Desktop\Semester project\exports

📊 Prime righe del CSV:
       CoordinateX  CoordinateY  CoefPressure
34752     0.990000     0.000627      0.003140
34753     0.990000     0.001046     -0.025741
34754     0.990000     0.001046     -0.025741
34755     0.978923     0.002955      0.014884
34756     0.967861     0.004787     -0.014976


In [10]:
import os

# Cartella dove salvare il CSV
export_dir = r"C:\Users\Lorenzo\Desktop\Semester project\exports"
os.makedirs(export_dir, exist_ok=True)

# ===========================
# PARAMETRI DA SCEGLIERE
# ===========================
case_number = 4        # il numero del caso che vuoi salvare
slice_number = 8       # numero dello slice

# ===========================
# SELEZIONE DELLO SLICE 2D
# ===========================
slice_df = df_2d.loc[
    (df_2d['case_num'] == case_number) &
    (df_2d['slice_num'] == slice_number),
    ['CoordinateX', 'CoordinateY', 'CoefPressure']
]

filename = f"airfoil_coords_pressure_2d_case{case_number}_slice{slice_number}.csv"

# ===========================
# SALVATAGGIO
# ===========================
slice_df.to_csv(os.path.join(export_dir, filename), index=False)

print(f"✅ Slice 2D con pressione salvata in CSV: {filename}")
print("📁 Percorso cartella:", export_dir)
print("\n📊 Prime righe del CSV:")
print(slice_df.head())


✅ Slice 2D con pressione salvata in CSV: airfoil_coords_pressure_2d_case4_slice8.csv
📁 Percorso cartella: C:\Users\Lorenzo\Desktop\Semester project\exports

📊 Prime righe del CSV:
       CoordinateX  CoordinateY  CoefPressure
34752     0.990000     0.000627      0.003140
34753     0.990000     0.001046     -0.025741
34754     0.990000     0.001046     -0.025741
34755     0.978923     0.002955      0.014884
34756     0.967861     0.004787     -0.014976


In [3]:
# Quanti case_num unici ci sono
n_cases = df_2d['case_num'].nunique()

# Quanti slice_num unici ci sono in totale (su tutti i case)
n_slices_total = df_2d['slice_num'].nunique()

# Quanti slice per ogni caso
slices_per_case = df_2d.groupby('case_num')['slice_num'].nunique()

print(f"🔹 Numero totale di case_num: {n_cases}")
print(f"🔹 Numero totale di slice_num (unici nel dataset): {n_slices_total}")
print("\n📊 Numero di slice per ciascun case_num:")
print(slices_per_case.head(10))  # mostra solo i primi 10 per non allungare troppo


🔹 Numero totale di case_num: 935
🔹 Numero totale di slice_num (unici nel dataset): 822

📊 Numero di slice per ciascun case_num:
case_num
0.0      45
1.0      62
2.0      65
4.0      74
5.0     216
6.0      47
8.0      55
9.0      53
10.0     41
11.0     54
Name: slice_num, dtype: int64


In [4]:
import os
import pandas as pd

# ===================================
# PARAMETRI
# ===================================
case_number = 0
num_slices_to_save = 5
base_path = r"C:\Users\Lorenzo\Desktop\Semester project\exports\Slices"
case_folder = os.path.join(base_path, f"Case_{case_number}")

# Crea la cartella se non esiste
os.makedirs(case_folder, exist_ok=True)

# ===================================
# TROVA GLI SLICE DISPONIBILI
# ===================================
slice_ids = sorted(df_2d.loc[df_2d['case_num'] == case_number, 'slice_num'].unique())
selected_slices = slice_ids[:num_slices_to_save]
print(f"📊 Case {case_number} ha {len(slice_ids)} slice totali. Esporto i primi {len(selected_slices)}: {selected_slices}")

# ===================================
# SALVATAGGIO CSV
# ===================================
for slice_id in selected_slices:
    slice_df = df_2d.loc[
        (df_2d['case_num'] == case_number) &
        (df_2d['slice_num'] == slice_id),
        ['CoordinateX', 'CoordinateY', 'CoefPressure']
    ]
    
    filename = f"case{case_number}_slice{int(slice_id)}.csv"
    file_path = os.path.join(case_folder, filename)
    
    slice_df.to_csv(file_path, index=False)
    print(f"✅ Salvato: {file_path}")

print("\n🎉 Esportazione completata!")


📊 Case 0 ha 45 slice totali. Esporto i primi 5: [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]
✅ Salvato: C:\Users\Lorenzo\Desktop\Semester project\exports\Slices\Case_0\case0_slice0.csv
✅ Salvato: C:\Users\Lorenzo\Desktop\Semester project\exports\Slices\Case_0\case0_slice1.csv
✅ Salvato: C:\Users\Lorenzo\Desktop\Semester project\exports\Slices\Case_0\case0_slice2.csv
✅ Salvato: C:\Users\Lorenzo\Desktop\Semester project\exports\Slices\Case_0\case0_slice3.csv
✅ Salvato: C:\Users\Lorenzo\Desktop\Semester project\exports\Slices\Case_0\case0_slice4.csv

🎉 Esportazione completata!


In [5]:
import os
import pandas as pd

# ===================================
# PARAMETRI
# ===================================
case_number = 1
num_slices_to_save = 5
base_path = r"C:\Users\Lorenzo\Desktop\Semester project\exports\Slices"
case_folder = os.path.join(base_path, f"Case_{case_number}")

# Crea la cartella se non esiste
os.makedirs(case_folder, exist_ok=True)

# ===================================
# TROVA GLI SLICE DISPONIBILI
# ===================================
slice_ids = sorted(df_2d.loc[df_2d['case_num'] == case_number, 'slice_num'].unique())
selected_slices = slice_ids[:num_slices_to_save]
print(f"📊 Case {case_number} ha {len(slice_ids)} slice totali. Esporto i primi {len(selected_slices)}: {selected_slices}")

# ===================================
# SALVATAGGIO CSV
# ===================================
for slice_id in selected_slices:
    slice_df = df_2d.loc[
        (df_2d['case_num'] == case_number) &
        (df_2d['slice_num'] == slice_id),
        ['CoordinateX', 'CoordinateY', 'CoefPressure']
    ]
    
    filename = f"case{case_number}_slice{int(slice_id)}.csv"
    file_path = os.path.join(case_folder, filename)
    
    slice_df.to_csv(file_path, index=False)
    print(f"✅ Salvato: {file_path}")

print("\n🎉 Esportazione completata!")


📊 Case 1 ha 62 slice totali. Esporto i primi 5: [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]
✅ Salvato: C:\Users\Lorenzo\Desktop\Semester project\exports\Slices\Case_1\case1_slice0.csv
✅ Salvato: C:\Users\Lorenzo\Desktop\Semester project\exports\Slices\Case_1\case1_slice1.csv
✅ Salvato: C:\Users\Lorenzo\Desktop\Semester project\exports\Slices\Case_1\case1_slice2.csv
✅ Salvato: C:\Users\Lorenzo\Desktop\Semester project\exports\Slices\Case_1\case1_slice3.csv
✅ Salvato: C:\Users\Lorenzo\Desktop\Semester project\exports\Slices\Case_1\case1_slice4.csv

🎉 Esportazione completata!


In [3]:
import os
import pandas as pd

# ===================================
# PARAMETRI BASE
# ===================================
export_base = r"C:\Users\Lorenzo\Desktop\Semester project\exports\Cases"
os.makedirs(export_base, exist_ok=True)

# Trova tutti i case_num unici
all_cases = sorted(df_2d['case_num'].unique())

print(f"🔍 Trovati {len(all_cases)} case_num totali")

# ===================================
# LOOP SU OGNI CASE
# ===================================
for case_number in all_cases:
    case_folder = os.path.join(export_base, f"case_{int(case_number)}")
    os.makedirs(case_folder, exist_ok=True)

    # Trova tutti gli slice per questo case
    slice_ids = sorted(df_2d.loc[df_2d['case_num'] == case_number, 'slice_num'].unique())
    print(f"📁 Case {int(case_number)} → {len(slice_ids)} slices")

    for slice_id in slice_ids:
        slice_df = df_2d.loc[
            (df_2d['case_num'] == case_number) &
            (df_2d['slice_num'] == slice_id),
            ['CoordinateX', 'CoordinateY', 'CoefPressure']
        ]

        filename = f"slice_{int(slice_id)}.csv"
        file_path = os.path.join(case_folder, filename)

        slice_df.to_csv(file_path, index=False)

    print(f"✅ Case {int(case_number)} completato ({len(slice_ids)} slice salvati)\n")

print("\n🎉 Tutti i case esportati con successo!")


🔍 Trovati 935 case_num totali
📁 Case 0 → 45 slices
✅ Case 0 completato (45 slice salvati)

📁 Case 1 → 62 slices
✅ Case 1 completato (62 slice salvati)

📁 Case 2 → 65 slices
✅ Case 2 completato (65 slice salvati)

📁 Case 4 → 74 slices
✅ Case 4 completato (74 slice salvati)

📁 Case 5 → 216 slices
✅ Case 5 completato (216 slice salvati)

📁 Case 6 → 47 slices
✅ Case 6 completato (47 slice salvati)

📁 Case 8 → 55 slices
✅ Case 8 completato (55 slice salvati)

📁 Case 9 → 53 slices
✅ Case 9 completato (53 slice salvati)

📁 Case 10 → 41 slices
✅ Case 10 completato (41 slice salvati)

📁 Case 11 → 54 slices
✅ Case 11 completato (54 slice salvati)

📁 Case 12 → 88 slices
✅ Case 12 completato (88 slice salvati)

📁 Case 13 → 88 slices
✅ Case 13 completato (88 slice salvati)

📁 Case 14 → 83 slices
✅ Case 14 completato (83 slice salvati)

📁 Case 15 → 78 slices
✅ Case 15 completato (78 slice salvati)

📁 Case 16 → 59 slices
✅ Case 16 completato (59 slice salvati)

📁 Case 18 → 274 slices
✅ Case 18 comple